In [1]:
!pip -q install datasets pandas pyarrow

In [3]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

PROJECT_PATH = "/content/drive/MyDrive/Knowledge_Augmented_Educational_QA"

DATA_PATH = os.path.join(PROJECT_PATH, "data")

os.makedirs(DATA_PATH, exist_ok=True)

print("Project folder created successfully!")
print(DATA_PATH)

Project folder created successfully!
/content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data


In [5]:
from datasets import load_dataset

# Download the CK-12 TQA dataset
dataset = load_dataset("notefill/ck12-tqa-instruction")

print(dataset)

README.md:   0%|          | 0.00/13.3k [00:00<?, ?B/s]

ck12_tqa_train.jsonl:   0%|          | 0.00/9.14M [00:00<?, ?B/s]

ck12_tqa_val.jsonl:   0%|          | 0.00/3.24M [00:00<?, ?B/s]

ck12_tqa_test.jsonl:   0%|          | 0.00/3.51M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15154 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5309 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5797 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'split', 'lesson_id', 'lesson_name', 'has_diagram', 'instruction', 'input', 'output', 'question_type', 'question_subtype', 'options', 'option_labels', 'image_path'],
        num_rows: 15154
    })
    validation: Dataset({
        features: ['id', 'source', 'split', 'lesson_id', 'lesson_name', 'has_diagram', 'instruction', 'input', 'output', 'question_type', 'question_subtype', 'options', 'option_labels', 'image_path'],
        num_rows: 5309
    })
    test: Dataset({
        features: ['id', 'source', 'split', 'lesson_id', 'lesson_name', 'has_diagram', 'instruction', 'input', 'output', 'question_type', 'question_subtype', 'options', 'option_labels', 'image_path'],
        num_rows: 5797
    })
})


In [6]:
import pandas as pd

# Convert training split into a DataFrame
df = dataset["train"].to_pandas()

# Display first five records
display(df.head())

,id,source,split,lesson_id,lesson_name,has_diagram,instruction,input,output,question_type,question_subtype,options,option_labels,image_path
0,NDQ_000046,TQA-CK12,train,L_0002,earth science and its branches,False,Answer the following multiple choice question ...,Earth science is the study of\n\nOptions:\na) ...,d,Multiple Choice,Multiple Choice,"[solid Earth., Earths oceans., Earths atmosphe...","[a, b, c, d]",None
1,NDQ_000047,TQA-CK12,train,L_0002,earth science and its branches,False,Answer the following multiple choice question ...,A geologist would be most likely to investigat...,a,Multiple Choice,Multiple Choice,"[mountains form., people cause pollution., tor...","[a, b, c, d]",None
2,NDQ_000048,TQA-CK12,train,L_0002,earth science and its branches,False,Answer the following multiple choice question ...,Which type of Earth scientist might look for p...,c,Multiple Choice,Multiple Choice,"[meteorologist, climatologist, geologist, ecol...","[a, b, c, d]",None
3,NDQ_000049,TQA-CK12,train,L_0002,earth science and its branches,False,Answer the following multiple choice question ...,Chemical oceanography is the study of the\n\nO...,b,Multiple Choice,Multiple Choice,"[human pollution of ocean water., naturally oc...","[a, b, c, d]",None
4,NDQ_000050,TQA-CK12,train,L_0002,earth science and its branches,False,Answer the following multiple choice question ...,The problem of global warming is most likely t...,d,Multiple Choice,Multiple Choice,"[planetary geologist., seismologist., physical...","[a, b, c, d]",None


In [7]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Dataset shape: (15154, 14)

Column names:
['id', 'source', 'split', 'lesson_id', 'lesson_name', 'has_diagram', 'instruction', 'input', 'output', 'question_type', 'question_subtype', 'options', 'option_labels', 'image_path']

Data types:
id                  object
source              object
split               object
lesson_id           object
lesson_name         object
has_diagram           bool
instruction         object
input               object
output              object
question_type       object
question_subtype    object
options             object
option_labels       object
image_path          object
dtype: object


In [8]:
for split_name in dataset.keys():
    print(split_name, ":", len(dataset[split_name]))

for split_name in dataset.keys():

    temp_df = dataset[split_name].to_pandas()

    print(f"\n{split_name.upper()} SPLIT")
    print(temp_df["has_diagram"].value_counts())

    print("\nPercentages:")
    print(
        temp_df["has_diagram"]
        .value_counts(normalize=True) * 100
    )

train : 15154
validation : 5309
test : 5797

TRAIN SPLIT
has_diagram
False    8653
True     6501
Name: count, dtype: int64

Percentages:
has_diagram
False    57.100436
True     42.899564
Name: proportion, dtype: float64

VALIDATION SPLIT
has_diagram
True     2781
False    2528
Name: count, dtype: int64

Percentages:
has_diagram
True     52.382746
False    47.617254
Name: proportion, dtype: float64

TEST SPLIT
has_diagram
True     3285
False    2512
Name: count, dtype: int64

Percentages:
has_diagram
True     56.667242
False    43.332758
Name: proportion, dtype: float64


In [9]:
sample = df.iloc[0]

print("Question ID:", sample["id"])
print("Lesson:", sample["lesson_name"])
print("Has Diagram:", sample["has_diagram"])

print("\nQuestion:")
print(sample["input"])

print("\nOptions:")
print(sample["options"])

print("\nCorrect Answer:")
print(sample["output"])

Question ID: NDQ_000046
Lesson: earth science and its branches
Has Diagram: False

Question:
Earth science is the study of

Options:
a) solid Earth.
b) Earths oceans.
c) Earths atmosphere.
d) all of the above

Options:
['solid Earth.' 'Earths oceans.' 'Earths atmosphere.' 'all of the above']

Correct Answer:
d


In [10]:
from datasets import DatasetDict

# Keep only questions that do not require diagrams
text_only_dataset = DatasetDict({
    split_name: dataset[split_name].filter(
        lambda example: example["has_diagram"] == False
    )
    for split_name in dataset.keys()
})

# Verify the new dataset sizes
for split_name in text_only_dataset.keys():
    print(
        split_name,
        ":",
        len(text_only_dataset[split_name])
    )

Filter:   0%|          | 0/15154 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5309 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5797 [00:00<?, ? examples/s]

train : 8653
validation : 2528
test : 2512


In [11]:
for split_name in text_only_dataset.keys():

    temp_df = text_only_dataset[split_name].to_pandas()

    print(f"\n{split_name.upper()}")
    print(temp_df["has_diagram"].value_counts())

    print("Missing values:")
    print(temp_df.isnull().sum())


TRAIN
has_diagram
False    8653
Name: count, dtype: int64
Missing values:
id                     0
source                 0
split                  0
lesson_id              0
lesson_name            0
has_diagram            0
instruction            0
input                  0
output                 0
question_type          0
question_subtype       0
options                0
option_labels          0
image_path          8653
dtype: int64

VALIDATION
has_diagram
False    2528
Name: count, dtype: int64
Missing values:
id                     0
source                 0
split                  0
lesson_id              0
lesson_name            0
has_diagram            0
instruction            0
input                  0
output                 0
question_type          0
question_subtype       0
options                0
option_labels          0
image_path          2528
dtype: int64

TEST
has_diagram
False    2512
Name: count, dtype: int64
Missing values:
id                     0
source              

In [12]:
for split_name in text_only_dataset.keys():

    temp_df = text_only_dataset[split_name].to_pandas()

    print(
        split_name,
        "Non-empty image paths:",
        temp_df["image_path"].notna().sum()
    )

train Non-empty image paths: 0
validation Non-empty image paths: 0
test Non-empty image paths: 0


In [13]:
import os

FILTERED_PATH = os.path.join(
    DATA_PATH,
    "ck12_tqa_text_only"
)

# Save the complete DatasetDict, preserving all splits
text_only_dataset.save_to_disk(FILTERED_PATH)

print("Filtered dataset saved successfully!")
print(FILTERED_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/8653 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2528 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2512 [00:00<?, ? examples/s]

Filtered dataset saved successfully!
/content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/ck12_tqa_text_only


In [14]:
for split_name in text_only_dataset.keys():

    temp_df = text_only_dataset[split_name].to_pandas()

    file_path = os.path.join(
        DATA_PATH,
        f"{split_name}_text_only.csv"
    )

    temp_df.to_csv(file_path, index=False)

    print("Saved:", file_path)

Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/train_text_only.csv
Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/validation_text_only.csv
Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/test_text_only.csv


In [15]:
for split_name in text_only_dataset.keys():

    temp_df = text_only_dataset[split_name].to_pandas()

    print(f"\n{split_name.upper()}")
    print("Total questions:", len(temp_df))
    print("Diagram-based:", temp_df["has_diagram"].sum())
    print("Missing questions:", temp_df["input"].isna().sum())
    print("Missing answers:", temp_df["output"].isna().sum())
    print("Non-empty image paths:", temp_df["image_path"].notna().sum())


TRAIN
Total questions: 8653
Diagram-based: 0
Missing questions: 0
Missing answers: 0
Non-empty image paths: 0

VALIDATION
Total questions: 2528
Diagram-based: 0
Missing questions: 0
Missing answers: 0
Non-empty image paths: 0

TEST
Total questions: 2512
Diagram-based: 0
Missing questions: 0
Missing answers: 0
Non-empty image paths: 0


In [16]:
# Combine the lesson information from all three filtered splits

all_questions_df = pd.concat([
    text_only_dataset[split].to_pandas()
    for split in text_only_dataset.keys()
], ignore_index=True)

# Count unique lessons
unique_lessons = all_questions_df[
    ["lesson_id", "lesson_name"]
].drop_duplicates()

print("Total text-only questions:", len(all_questions_df))

print("Unique lessons:", len(unique_lessons))

print("\nFirst 15 lessons:")
display(unique_lessons.head(15))

for split_name in text_only_dataset.keys():

    temp_df = text_only_dataset[split_name].to_pandas()

    print(
        split_name,
        ":",
        temp_df["lesson_id"].nunique(),
        "unique lessons"
    )

Total text-only questions: 13693
Unique lessons: 1073

First 15 lessons:


,lesson_id,lesson_name
0,L_0002,earth science and its branches
24,L_0003,erosion and deposition by flowing water
48,L_0004,erosion and deposition by waves
72,L_0006,erosion and deposition by glaciers
96,L_0008,fossils
129,L_0009,relative ages of rocks
163,L_0010,absolute ages of rocks
197,L_0011,the origin of earth
231,L_0012,early earth
265,L_0014,water on earth


train : 664 unique lessons
validation : 199 unique lessons
test : 210 unique lessons


In [17]:
# Inspect one complete record
sample = text_only_dataset["train"][0]

for key, value in sample.items():
    print(f"\n{key}:")
    print(value)

print("Unique lessons across all splits:",
      unique_lessons["lesson_id"].nunique())

print("\nMissing lesson IDs:")
print(unique_lessons["lesson_id"].isna().sum())

print("\nDuplicate lesson IDs:")
print(unique_lessons["lesson_id"].duplicated().sum())


id:
NDQ_000046

source:
TQA-CK12

split:
train

lesson_id:
L_0002

lesson_name:
earth science and its branches

has_diagram:
False

instruction:
Answer the following multiple choice question from a science textbook.

input:
Earth science is the study of

Options:
a) solid Earth.
b) Earths oceans.
c) Earths atmosphere.
d) all of the above

output:
d

question_type:
Multiple Choice

question_subtype:
Multiple Choice

options:
['solid Earth.', 'Earths oceans.', 'Earths atmosphere.', 'all of the above']

option_labels:
['a', 'b', 'c', 'd']

image_path:
None
Unique lessons across all splits: 1073

Missing lesson IDs:
0

Duplicate lesson IDs:
0


In [18]:
knowledge_df = text_only_dataset["train"].to_pandas()

knowledge_df = knowledge_df[
    ["id", "lesson_id", "lesson_name", "input", "options", "output"]
].copy()

print("Knowledge corpus shape:", knowledge_df.shape)
knowledge_df.head()

Knowledge corpus shape: (8653, 6)


,id,lesson_id,lesson_name,input,options,output
0,NDQ_000046,L_0002,earth science and its branches,Earth science is the study of\n\nOptions:\na) ...,"[solid Earth., Earths oceans., Earths atmosphe...",d
1,NDQ_000047,L_0002,earth science and its branches,A geologist would be most likely to investigat...,"[mountains form., people cause pollution., tor...",a
2,NDQ_000048,L_0002,earth science and its branches,Which type of Earth scientist might look for p...,"[meteorologist, climatologist, geologist, ecol...",c
3,NDQ_000049,L_0002,earth science and its branches,Chemical oceanography is the study of the\n\nO...,"[human pollution of ocean water., naturally oc...",b
4,NDQ_000050,L_0002,earth science and its branches,The problem of global warming is most likely t...,"[planetary geologist., seismologist., physical...",d


In [19]:
knowledge_df = text_only_dataset["train"].to_pandas()[
    ["id", "lesson_id", "lesson_name",
     "input", "options", "option_labels", "output"]
].copy()

print(knowledge_df.shape)

(8653, 7)


In [20]:
def create_knowledge_entry(row):

    answer_label = row["output"]

    answer_index = list(row["option_labels"]).index(answer_label)
    correct_answer = row["options"][answer_index]

    knowledge_entry = f"""
Lesson: {row['lesson_name']}

{row['input']}

Correct Answer: {answer_label}) {correct_answer}
"""

    return knowledge_entry.strip()

sample_entry = create_knowledge_entry(knowledge_df.iloc[0])

print(sample_entry)

Lesson: earth science and its branches

Earth science is the study of

Options:
a) solid Earth.
b) Earths oceans.
c) Earths atmosphere.
d) all of the above

Correct Answer: d) all of the above


In [21]:
mismatched_rows = []

for idx, row in knowledge_df.iterrows():

    labels = list(row["option_labels"])
    answer = str(row["output"]).strip()

    if answer not in labels:
        mismatched_rows.append(idx)

print("Total mismatched rows:", len(mismatched_rows))

if mismatched_rows:
    display(
        knowledge_df.loc[
            mismatched_rows[:5],
            ["id", "lesson_name", "options",
             "option_labels", "output"]
        ]
    )

Total mismatched rows: 10


,id,lesson_name,options,option_labels,output
589,NDQ_001257,climate change,"[large asteroid striking Earth., huge volcanic...","[a, b, c]",d
2156,NDQ_004158,distribution of water on earth,"[true, false]","[c, d]",b
2254,NDQ_004388,effects of air pollution on human health,"[true, false]","[c, d]",a
2485,NDQ_004717,greenhouse effect,"[true, false]","[c, d]",a
2514,NDQ_004749,groundwater pollution,"[true, false]","[c, d]",b


In [22]:
for idx in mismatched_rows:
    row = knowledge_df.loc[idx]

    print("\n" + "=" * 60)
    print("ID:", row["id"])
    print("Lesson:", row["lesson_name"])
    print("Question:", row["input"])
    print("Options:", row["options"])
    print("Labels:", row["option_labels"])
    print("Output:", row["output"])


ID: NDQ_001257
Lesson: climate change
Question: Natural processes that may have affected Earths temperature in the past include a ##change in the tilt of Earths axis.

Options:
a) large asteroid striking Earth.
b) huge volcanic eruption.
c) all of the above
Options: ['large asteroid striking Earth.' 'huge volcanic eruption.'
 'all of the above']
Labels: ['a' 'b' 'c']
Output: d

ID: NDQ_004158
Lesson: distribution of water on earth
Question: all of earths water supply is accessible for us to use.

Options:
c) true
d) false
Options: ['true' 'false']
Labels: ['c' 'd']
Output: b

ID: NDQ_004388
Lesson: effects of air pollution on human health
Question: breathing air into which weve emitted pollutants is not much different from throwing trash in our food supply and eating it.

Options:
c) true
d) false
Options: ['true' 'false']
Labels: ['c' 'd']
Output: a

ID: NDQ_004717
Lesson: greenhouse effect
Question: atmospheric greenhouse gas levels are increasing due to human activities.

Options:


In [25]:
# Keep only entries with matching answer labels

valid_indices = []

for idx, row in knowledge_df.iterrows():

    labels = list(row["option_labels"])
    answer = str(row["output"]).strip()

    if answer in labels:
        valid_indices.append(idx)

clean_knowledge_df = knowledge_df.loc[valid_indices].copy()

print("Original entries:", len(knowledge_df))
print("Clean entries:", len(clean_knowledge_df))
print("Excluded entries:", len(knowledge_df) - len(clean_knowledge_df))
clean_knowledge_df["knowledge_text"] = clean_knowledge_df.apply(
    create_knowledge_entry, axis=1
)

print(clean_knowledge_df["knowledge_text"].iloc[0])

Original entries: 8653
Clean entries: 8643
Excluded entries: 10
Lesson: earth science and its branches

Earth science is the study of

Options:
a) solid Earth.
b) Earths oceans.
c) Earths atmosphere.
d) all of the above

Correct Answer: d) all of the above


In [26]:
CLEAN_CORPUS_PATH = os.path.join(
    DATA_PATH, "clean_knowledge_corpus.csv"
)

clean_knowledge_df.to_csv(
    CLEAN_CORPUS_PATH,
    index=False
)

print("Saved successfully:", CLEAN_CORPUS_PATH)

Saved successfully: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/clean_knowledge_corpus.csv


In [27]:
# Save cleaned training knowledge corpus
CLEAN_CORPUS_PATH = os.path.join(
    DATA_PATH, "clean_knowledge_corpus.csv"
)

clean_knowledge_df.to_csv(CLEAN_CORPUS_PATH, index=False)

# Save excluded entries separately
EXCLUDED_PATH = os.path.join(
    DATA_PATH, "excluded_inconsistent_entries.csv"
)

excluded_df = knowledge_df.drop(
    index=clean_knowledge_df.index
)

excluded_df.to_csv(EXCLUDED_PATH, index=False)

print("Clean corpus saved:", CLEAN_CORPUS_PATH)
print("Excluded entries saved:", EXCLUDED_PATH)
print("Clean corpus shape:", clean_knowledge_df.shape)
print("Excluded entries shape:", excluded_df.shape)

Clean corpus saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/clean_knowledge_corpus.csv
Excluded entries saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/excluded_inconsistent_entries.csv
Clean corpus shape: (8643, 8)
Excluded entries shape: (10, 7)


In [28]:
for split_name in ["validation", "test"]:

    split_df = text_only_dataset[split_name].to_pandas()

    mismatches = []

    for idx, row in split_df.iterrows():

        labels = list(row["option_labels"])
        answer = str(row["output"]).strip()

        if answer not in labels:
            mismatches.append(idx)

    print(f"\n{split_name.upper()}")
    print("Total entries:", len(split_df))
    print("Mismatched entries:", len(mismatches))
    print("Valid entries:", len(split_df) - len(mismatches))


VALIDATION
Total entries: 2528
Mismatched entries: 4
Valid entries: 2524

TEST
Total entries: 2512
Mismatched entries: 4
Valid entries: 2508


In [29]:
for split_name in ["validation", "test"]:

    split_df = text_only_dataset[split_name].to_pandas()

    print("\n", "=" * 60)
    print(split_name.upper())

    for idx, row in split_df.iterrows():

        labels = list(row["option_labels"])
        answer = str(row["output"]).strip()

        if answer not in labels:
            print("\nID:", row["id"])
            print("Lesson:", row["lesson_name"])
            print("Question:", row["input"])
            print("Options:", row["options"])
            print("Labels:", row["option_labels"])
            print("Output:", row["output"])


VALIDATION

ID: NDQ_004257
Lesson: earthquakes at convergent plate boundaries
Question: the damage caused by earthquakes that took place a century or more ago is a good indication of the damage a similar quake would cause today.

Options:
e) true
f) false
Options: ['true' 'false']
Labels: ['e' 'f']
Output: b

ID: NDQ_005966
Lesson: soil formation
Question: steep slopes tend to develop rich soils.

Options:
c) true
d) false
Options: ['true' 'false']
Labels: ['c' 'd']
Output: b

ID: NDQ_005979
Lesson: soil horizons and profiles
Question: in soil, water is held by carbonate minerals.

Options:
c) true
d) false
Options: ['true' 'false']
Labels: ['c' 'd']
Output: b

ID: NDQ_006280
Lesson: types of soils
Question: soil forms at roughly the same rate in all climate types.

Options:
c) true
d) false
Options: ['true' 'false']
Labels: ['c' 'd']
Output: b

TEST

ID: NDQ_004559
Lesson: flow of matter in ecosystems
Question: humans are not part of food chains and food webs because we get most of o

In [30]:
from datasets import DatasetDict

clean_splits = {}
excluded_splits = {}

for split_name in ["validation", "test"]:

    split_df = text_only_dataset[split_name].to_pandas()

    valid_mask = split_df.apply(
        lambda row: str(row["output"]).strip()
        in list(row["option_labels"]),
        axis=1
    )

    clean_splits[split_name] = split_df[valid_mask].copy()
    excluded_splits[split_name] = split_df[~valid_mask].copy()

    print(
        split_name,
        "Valid:", len(clean_splits[split_name]),
        "Excluded:", len(excluded_splits[split_name])
    )

validation Valid: 2524 Excluded: 4
test Valid: 2508 Excluded: 4


In [31]:
for split_name in ["validation", "test"]:

    clean_path = os.path.join(
        DATA_PATH, f"{split_name}_clean.csv"
    )

    excluded_path = os.path.join(
        DATA_PATH, f"{split_name}_excluded.csv"
    )

    clean_splits[split_name].to_csv(clean_path, index=False)
    excluded_splits[split_name].to_csv(excluded_path, index=False)

    print("Saved:", clean_path)
    print("Saved:", excluded_path)

Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/validation_clean.csv
Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/validation_excluded.csv
Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/test_clean.csv
Saved: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/test_excluded.csv


In [32]:
# Check all three cleaned datasets

datasets_to_check = {
    "train": clean_knowledge_df,
    "validation": clean_splits["validation"],
    "test": clean_splits["test"]
}

for name, df in datasets_to_check.items():

    print("\n", "=" * 45)
    print(name.upper())

    print("Rows:", len(df))
    print("Duplicate IDs:", df["id"].duplicated().sum())
    print("Missing IDs:", df["id"].isna().sum())
    print("Missing questions:", df["input"].isna().sum())
    print("Missing answers:", df["output"].isna().sum())

    print(
        "Answer-label mismatches:",
        sum(
            str(row["output"]).strip()
            not in list(row["option_labels"])
            for _, row in df.iterrows()
        )
    )


TRAIN
Rows: 8643
Duplicate IDs: 0
Missing IDs: 0
Missing questions: 0
Missing answers: 0
Answer-label mismatches: 0

VALIDATION
Rows: 2524
Duplicate IDs: 0
Missing IDs: 0
Missing questions: 0
Missing answers: 0
Answer-label mismatches: 0

TEST
Rows: 2508
Duplicate IDs: 0
Missing IDs: 0
Missing questions: 0
Missing answers: 0
Answer-label mismatches: 0


In [33]:
splits = {
    "train": clean_knowledge_df,
    "validation": clean_splits["validation"],
    "test": clean_splits["test"]
}

# Collect question IDs and lesson IDs
id_sets = {
    name: set(df["id"])
    for name, df in splits.items()
}

lesson_sets = {
    name: set(df["lesson_id"])
    for name, df in splits.items()
}

# Check every pair of splits
pairs = [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]

for a, b in pairs:
    print(f"\n{a.upper()} vs {b.upper()}")
    print("Overlapping question IDs:", len(id_sets[a] & id_sets[b]))
    print("Overlapping lesson IDs:", len(lesson_sets[a] & lesson_sets[b]))


TRAIN vs VALIDATION
Overlapping question IDs: 0
Overlapping lesson IDs: 0

TRAIN vs TEST
Overlapping question IDs: 0
Overlapping lesson IDs: 0

VALIDATION vs TEST
Overlapping question IDs: 0
Overlapping lesson IDs: 0


In [34]:
!pip -q install transformers

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Model:", MODEL_NAME)
print("Tokenizer maximum length:", tokenizer.model_max_length)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!
Model: Qwen/Qwen2.5-7B-Instruct
Tokenizer maximum length: 131072


In [35]:
import numpy as np

knowledge_texts = clean_knowledge_df["knowledge_text"].tolist()

token_counts = []

for i in range(0, len(knowledge_texts), 500):
    batch = knowledge_texts[i:i+500]

    encoded = tokenizer(
        batch,
        add_special_tokens=False,
        truncation=False
    )

    token_counts.extend([
        len(ids) for ids in encoded["input_ids"]
    ])

token_counts = np.array(token_counts)

print("Number of knowledge entries:", len(token_counts))
print("Total tokens:", token_counts.sum())
print("Average tokens per entry:", round(token_counts.mean(), 2))
print("Median tokens per entry:", np.median(token_counts))
print("90th percentile:", np.percentile(token_counts, 90))
print("95th percentile:", np.percentile(token_counts, 95))
print("99th percentile:", np.percentile(token_counts, 99))
print("Maximum tokens in one entry:", token_counts.max())

Number of knowledge entries: 8643
Total tokens: 434663
Average tokens per entry: 50.29
Median tokens per entry: 48.0
90th percentile: 72.0
95th percentile: 81.0
99th percentile: 102.0
Maximum tokens in one entry: 175


In [36]:
readme_content = """
# CK-12 Textbook Question Answering Dataset

## 1. Dataset Information

**Dataset:** CK-12 Textbook Question Answering (TQA)

**Hugging Face version:** notefill/ck12-tqa-instruction

**Dataset link:**
https://huggingface.co/datasets/notefill/ck12-tqa-instruction

**Original dataset page:**
https://prior.allenai.org/projects/tqa

**Original research paper:**
Kembhavi et al. (2017).
Are You Smarter Than a Sixth Grader?
Textbook Question Answering for Multimodal Machine Comprehension.
CVPR 2017.

## 2. Research Usage

This dataset is used to evaluate knowledge-augmented
Large Language Models in an educational question-answering task.

The experiment compares four architectures:

1. LLM-only
2. LLM + Retrieval-Augmented Generation (RAG)
3. LLM + Cache-Augmented Generation (CAG)
4. LLM + RAG + CAG

The knowledge corpus is derived from training-split
question-answer entries.

Important: This is a question-answer-derived knowledge
corpus, not a collection of complete textbook lesson passages.

## 3. Dataset Filtering

The original Hugging Face dataset contains three splits:

- Training
- Validation
- Test

Only text-based questions were retained.

Questions requiring diagrams or images were excluded
using the has_diagram field.

### Text-only dataset counts

| Split | Records |
|---|---:|
| Training | 8,653 |
| Validation | 2,528 |
| Test | 2,512 |
| Total | 13,693 |

## 4. Data Cleaning

The answer label was checked against the available
option labels for every record.

Records with inconsistent answer labels were excluded.
No answer labels were guessed or manually corrected.

### Final cleaned dataset

| Split | Original | Retained | Excluded |
|---|---:|---:|---:|
| Training | 8,653 | 8,643 | 10 |
| Validation | 2,528 | 2,524 | 4 |
| Test | 2,512 | 2,508 | 4 |
| Total | 13,693 | 13,675 | 18 |

Retention rate: approximately 99.87%.

## 5. Integrity Validation

The cleaned splits were checked for:

- Duplicate question IDs
- Missing question IDs
- Missing questions
- Missing answers
- Answer-label inconsistencies
- Overlapping question IDs across splits
- Overlapping lesson IDs across splits

All checks passed.

There were zero overlapping question IDs and
zero overlapping lesson IDs between every pair of splits.

## 6. Knowledge Corpus Construction

The training split was used to construct the knowledge corpus.

Each knowledge entry contains:

- Lesson name
- Question
- Answer options
- Correct answer

The resulting knowledge corpus contains 8,643 entries.

Validation and test answers are retained for evaluation
and are not included in the training knowledge corpus.

## 7. Token Analysis

Tokenization was performed using the tokenizer of:

Qwen/Qwen2.5-7B-Instruct

| Metric | Result |
|---|---:|
| Knowledge entries | 8,643 |
| Total tokens | 434,663 |
| Average tokens per entry | 50.29 |
| Median tokens per entry | 48 |
| 90th percentile | 72 |
| 95th percentile | 81 |
| 99th percentile | 102 |
| Maximum tokens | 175 |

## 8. Files

- clean_knowledge_corpus.csv:
  Cleaned training knowledge entries.

- validation_clean.csv:
  Cleaned validation records.

- test_clean.csv:
  Cleaned test records.

- excluded_inconsistent_entries.csv:
  Training records excluded due to answer-label inconsistency.

- validation_excluded.csv:
  Excluded validation records.

- test_excluded.csv:
  Excluded test records.

- train_text_only.csv:
  Original text-only training records.

- validation_text_only.csv:
  Original text-only validation records.

- test_text_only.csv:
  Original text-only test records.

- ck12_tqa_text_only/:
  Saved text-only DatasetDict.

## 9. Reproducibility and License Note

The dataset was loaded from the Hugging Face dataset
notefill/ck12-tqa-instruction.

The original CK-12 TQA dataset and its reformatted
Hugging Face version should be treated as distinct
dataset distributions.

The original dataset's license information should be
verified before redistributing the dataset or its contents.
This project does not assume that the original and
reformatted versions have identical licensing terms.
"""

README_PATH = os.path.join(DATA_PATH, "README.md")

with open(README_PATH, "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Dataset README created successfully!")
print("Saved at:", README_PATH)

Dataset README created successfully!
Saved at: /content/drive/MyDrive/Knowledge_Augmented_Educational_QA/data/README.md


In [37]:
print(os.path.exists(README_PATH))
print(os.path.getsize(README_PATH), "bytes")

True
4071 bytes
